# Implementing my own PCA algorithm for Dimensionality Reduction

This notebook builds a small PCA implementation from scratch and compares it with scikit-learn's PCA on a student quiz dataset.  
Main technologies used here: Python, NumPy, Pandas, and scikit-learn (for comparison at the end).

The goal is to reduce the dimensionality of the dataset, inspect how much variance each principal component keeps, and check whether the custom implementation behaves the same way as the library version.


In [1]:
# Imports
import numpy as np
import pandas as pd

## I - Building the PCA class

In [2]:
class PCA_gui:
    """
    Basic PCA implementation built from scratch.

    Parameters
    ----------
    n_components : int
        Number of principal components to keep.

    Attributes
    ----------
    explained_var : numpy.ndarray or None
        Eigenvalues associated with the selected principal components.
    pdirs : numpy.ndarray or None
        Principal directions (eigenvectors) of the selected components.
    output : numpy.ndarray or None
        Data projected into the principal component space.
    cov_matrix : numpy.ndarray or None
        Covariance matrix computed from the centered input data.
    explained_var_ratio : numpy.ndarray or None
        Fraction of total variance explained by each selected component.
    cumulative_var : numpy.ndarray or None
        Cumulative explained variance across the selected components.

    Notes
    -----
    This class expects a 2D matrix with shape (n_samples, n_features).
    It centers the data, computes the covariance matrix, and then uses
    eigenvalue decomposition to extract the principal components.
    """

    def __init__(self, n_components):
        self.n_components = n_components

        self.explained_var = None
        self.pdirs = None
        self.output = None
        self.cov_matrix = None
        self.explained_var_ratio = None
        self.cumulative_var = None

    def fit(self, input_data):
        """
        Fit PCA to the input matrix.

        Parameters
        ----------
        input_data : array-like
            Input matrix with shape (n_samples, n_features).

        Raises
        ------
        TypeError
            If the input is not a 2D matrix.
        """
        # Accept lists/DataFrames too and convert them on the fly.
        if not isinstance(input_data, np.ndarray):
            input_data = np.array(input_data)

        # PCA expects a 2D matrix: rows are samples, columns are features.
        if len(input_data.shape) != 2:
            raise TypeError("The input must be a 2D matrix with shape (n_samples, n_features).")

        num_samples, num_features = input_data.shape

        # Center the data before computing the covariance matrix.
        centered_input = input_data - np.mean(input_data, axis=0)

        # Covariance matrix of the centered data.
        self.cov_matrix = np.dot(centered_input.T, centered_input) / (num_samples - 1)

        # eigh is appropriate here because the covariance matrix is symmetric.
        eigenvalues, eigenvectors = np.linalg.eigh(self.cov_matrix)

        # Sort from largest to smallest eigenvalue.
        sorted_idx = eigenvalues.argsort()[::-1]

        self.explained_var = eigenvalues[sorted_idx][:self.n_components]
        self.pdirs = eigenvectors[:, sorted_idx][:, :self.n_components]

        # Project the data onto the ordered principal directions.
        self.output = np.dot(centered_input, eigenvectors[:, sorted_idx])

        # Explained variance ratio is just the share of total variance.
        total_variance = np.sum(eigenvalues)
        if total_variance != 0:
            self.explained_var_ratio = self.explained_var / total_variance
            self.cumulative_var = np.cumsum(self.explained_var_ratio)
        else:
            print("Total variance is zero. The data matrix is probably linearly dependent.")


Here we will reduce the dimensionality of a dataset that contains the answers from **40 students** to a quiz with **49 questions**. Each answer represents one variable in the problem. The data is fictional and only used as an example.

The first column (which we will load as the index) contains each student's anonymous ID.


In [3]:
# Load the dataset
df = pd.read_csv("./data/dataset.csv", index_col=0)

In [4]:
# Quick look at the dataset size and first rows
print("Number of records (rows):", df.shape[0], "Number of features (columns):", df.shape[1])
df.head()


Number of records (rows): 40 Number of features (columns): 49


,186,295,321,337,464,469,502,506,563,931,...,1401,1402,1414,1422,1524,1551,1553,1559,1561,1568
NSCLC_A549_1,171210.5829,1246686.524,527193.4870,411155.8781,59929.70461,49255.07973,171273.1364,159183.1286,663694.9741,35070.49713,...,95049.95991,1.719464e+06,645473.1920,89954.00002,52983.82193,1.672322e+06,6.867305e+05,259889.4715,1.563879e+06,189971.51110
NSCLC_H1703_2,204751.3591,1338013.461,571379.0841,486137.0920,89261.66256,72052.72202,187464.2389,180000.6140,443440.3745,89938.26439,...,131002.12740,8.290090e+05,293504.7109,42264.73772,97472.88420,1.608167e+06,9.408252e+05,363398.5823,3.043114e+05,35762.79339
NSCLC_H1703_1,203558.4952,1040438.105,498460.6875,411052.8868,96002.36973,0.00000,193894.4078,179518.9387,736672.9068,107041.34120,...,108827.84800,8.541730e+05,313811.6234,51125.12753,95633.56928,1.931491e+06,1.086556e+06,417606.0810,4.365568e+05,53865.65208
NSCLC_A549_2,245859.2006,1371135.588,153050.9373,495539.7034,81436.65785,85158.56741,0.0000,186757.1440,809830.2341,43565.82607,...,106003.59820,2.037000e+06,762600.1361,116878.29910,58497.52991,2.077164e+06,8.461167e+05,328548.1739,1.824857e+06,243842.05260
NSCLC_H1437_1,214448.1780,1107105.986,524333.5670,484994.8797,92368.82235,69868.54980,181168.8533,0.0000,635990.2172,51087.42828,...,83218.09256,1.269266e+06,459612.8875,64837.50435,0.00000,9.434720e+05,4.022300e+05,153070.9443,3.745915e+05,44104.41364


In [5]:
# Convert the table to a NumPy array
data = df.to_numpy()

# A tiny sanity check before running PCA
print("Number of records (rows):", data.shape[0], "Number of features (columns):", data.shape[1])
print("Now the object type is:", type(data))

Number of records (rows): 40 Number of features (columns): 49
Now the object type is: <class 'numpy.ndarray'>


## II - Running our PCA class

In [6]:
n_components = 10
pca_gui = PCA_gui(n_components)

# Quick check: a 1D array should fail here, and that's expected.
pca_gui.fit(np.array([1, 2, 3]))


TypeError: The input must be a 2D matrix with shape (n_samples, n_features).

In [7]:
# Another quick check: a 3D array should also raise an error.
pca_gui.fit(np.array([[[1, 2, 3], [4, 5, 6]], [[7, 8, 9], [10, 11, 12]]]))


TypeError: The input must be a 2D matrix with shape (n_samples, n_features).

In [8]:
# Good, the validation above is doing its job. Now let's fit PCA on the actual data.
pca_gui.fit(data)

print("PCA Gui -> Explained variance (eigenvalues of cov. matrix):\n")
for i in range(n_components):
    print(f"PC{i+1} -> {pca_gui.explained_var[i]:.2e}\n")

PCA Gui -> Explained variance (eigenvalues of cov. matrix):

PC1 -> 1.35e+12

PC2 -> 5.01e+11

PC3 -> 4.37e+11

PC4 -> 3.20e+11

PC5 -> 2.16e+11

PC6 -> 1.28e+11

PC7 -> 9.68e+10

PC8 -> 7.48e+10

PC9 -> 5.49e+10

PC10 -> 4.29e+10



In [9]:
# Explained variance ratio for each principal component
print("PCA Gui -> Explained variance per principal component:\n")
for i in range(n_components):
    print(f"PC{i+1} -> {pca_gui.explained_var_ratio[i]:.2%}\n")
print("The first principal component explains ~40% of the variance of the data, the second ~15% and so on...")

PCA Gui -> Explained variance per principal component:

PC1 -> 40.06%

PC2 -> 14.91%

PC3 -> 13.00%

PC4 -> 9.53%

PC5 -> 6.44%

PC6 -> 3.81%

PC7 -> 2.88%

PC8 -> 2.23%

PC9 -> 1.64%

PC10 -> 1.28%

The first principal component explains ~40% of the variance of the data, the second ~15% and so on...


In [10]:
# Cumulative explained variance helps us decide how many components are worth keeping
print("PCA Gui -> Cumulative explained variance:\n")
# Define a helper string to format the answer in a pretty way :)
helper_string = ""
for i in range(n_components):
    for j in range(i):
        helper_string += f"PC{j+1} + "
    helper_string += f"PC{i+1}"
    print(f"{helper_string} -> {pca_gui.cumulative_var[i]:.2%}\n")
    helper_string = ""
print("Keeping only the first principal component ~40% of the data variance is explained, keeping the first and the second ~55% of the data variance is explained and so on ...")

PCA Gui -> Cumulative explained variance:

PC1 -> 40.06%

PC1 + PC2 -> 54.97%

PC1 + PC2 + PC3 -> 67.97%

PC1 + PC2 + PC3 + PC4 -> 77.50%

PC1 + PC2 + PC3 + PC4 + PC5 -> 83.94%

PC1 + PC2 + PC3 + PC4 + PC5 + PC6 -> 87.76%

PC1 + PC2 + PC3 + PC4 + PC5 + PC6 + PC7 -> 90.64%

PC1 + PC2 + PC3 + PC4 + PC5 + PC6 + PC7 + PC8 -> 92.87%

PC1 + PC2 + PC3 + PC4 + PC5 + PC6 + PC7 + PC8 + PC9 -> 94.51%

PC1 + PC2 + PC3 + PC4 + PC5 + PC6 + PC7 + PC8 + PC9 + PC10 -> 95.78%

Keeping only the first principal component ~40% of the data variance is explained, keeping the first and the second ~55% of the data variance is explained and so on ...


Around 96% of the information (variance) from the original 49 variables is represented by 10 principal components.  
That means we can use these 10 components as inputs for a machine learning model, for example, while keeping most of the useful structure in the data.


## III - Compare with scikit-learn's PCA

In [11]:
import sklearn
from sklearn.decomposition import PCA

In [12]:
# Apply scikit-learn's PCA
pca_skl = PCA(n_components)
x_pca = pca_skl.fit_transform(data)

In [13]:
# Explained variance for each component
print("PCA Sklearn -> Explained variance (eigenvalues of cov. matrix):\n")
for i in range(n_components):
    print(f"SKL_PC{i+1} -> {pca_skl.explained_variance_[i]:.2e}\n")
print("----> SAME RESULT AS THE PCA IMPLEMENTED FROM SCRATCH <----")

PCA Sklearn -> Explained variance (eigenvalues of cov. matrix):

SKL_PC1 -> 1.35e+12

SKL_PC2 -> 5.01e+11

SKL_PC3 -> 4.37e+11

SKL_PC4 -> 3.20e+11

SKL_PC5 -> 2.16e+11

SKL_PC6 -> 1.28e+11

SKL_PC7 -> 9.68e+10

SKL_PC8 -> 7.48e+10

SKL_PC9 -> 5.49e+10

SKL_PC10 -> 4.29e+10

----> SAME RESULT AS THE PCA IMPLEMENTED FROM SCRATCH <----


In [14]:
# Explained variance ratio for each principal component
print("PCA Sklearn -> Explained variance per principal component:\n")
for i in range(n_components):
    print(f"SKL_PC{i+1} -> {pca_skl.explained_variance_ratio_[i]:.2%}\n")
print("The first principal component explains ~40% of the variance of the data, the second ~15% and so on ...\n")
print("----> SAME RESULT AS THE PCA IMPLEMENTED FROM SCRATCH <----")

PCA Sklearn -> Explained variance per principal component:

SKL_PC1 -> 40.06%

SKL_PC2 -> 14.91%

SKL_PC3 -> 13.00%

SKL_PC4 -> 9.53%

SKL_PC5 -> 6.44%

SKL_PC6 -> 3.81%

SKL_PC7 -> 2.88%

SKL_PC8 -> 2.23%

SKL_PC9 -> 1.64%

SKL_PC10 -> 1.28%

The first principal component explains ~40% of the variance of the data, the second ~15% and so on ...

----> SAME RESULT AS THE PCA IMPLEMENTED FROM SCRATCH <----


In [15]:
# Cumulative explained variance
print("PCA Sklearn -> Cumulative explained variance:\n")
# As before, define a helper string to format the answer in a pretty way :)
helper_string = ""
for i in range(n_components):
    for j in range(i):
        helper_string += f"SKL_PC{j+1} + "
    helper_string += f"SKL_PC{i+1}"
    print(f"{helper_string} -> {np.cumsum(pca_skl.explained_variance_ratio_)[i]:.2%}\n")
    helper_string = ""
print("Keeping only the first principal component ~40% of the data variance is explained, keeping the first and the second ~55% of the data variance is explained and so on ...\n")
print("----> SAME RESULT AS THE PCA IMPLEMENTED FROM SCRATCH <----")

PCA Sklearn -> Cumulative explained variance:

SKL_PC1 -> 40.06%

SKL_PC1 + SKL_PC2 -> 54.97%

SKL_PC1 + SKL_PC2 + SKL_PC3 -> 67.97%

SKL_PC1 + SKL_PC2 + SKL_PC3 + SKL_PC4 -> 77.50%

SKL_PC1 + SKL_PC2 + SKL_PC3 + SKL_PC4 + SKL_PC5 -> 83.94%

SKL_PC1 + SKL_PC2 + SKL_PC3 + SKL_PC4 + SKL_PC5 + SKL_PC6 -> 87.76%

SKL_PC1 + SKL_PC2 + SKL_PC3 + SKL_PC4 + SKL_PC5 + SKL_PC6 + SKL_PC7 -> 90.64%

SKL_PC1 + SKL_PC2 + SKL_PC3 + SKL_PC4 + SKL_PC5 + SKL_PC6 + SKL_PC7 + SKL_PC8 -> 92.87%

SKL_PC1 + SKL_PC2 + SKL_PC3 + SKL_PC4 + SKL_PC5 + SKL_PC6 + SKL_PC7 + SKL_PC8 + SKL_PC9 -> 94.51%

SKL_PC1 + SKL_PC2 + SKL_PC3 + SKL_PC4 + SKL_PC5 + SKL_PC6 + SKL_PC7 + SKL_PC8 + SKL_PC9 + SKL_PC10 -> 95.78%

Keeping only the first principal component ~40% of the data variance is explained, keeping the first and the second ~55% of the data variance is explained and so on ...

----> SAME RESULT AS THE PCA IMPLEMENTED FROM SCRATCH <----


We conclude that our PCA yield exact same result as scikit-learn PCA algorithm! Right on!